# Customer Segmentation — Data Preparation & Preprocessing

**Project:** Customer Segmentation (Topic 8)  
**Goal:** Segment customers into distinct groups based on demographic, behavioural, and transactional data to allow businesses to tailor marketing strategies.

---





---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('seaborn-v0_8-whitegrid')

print('All libraries loaded successfully!')

---
## 2. Load Dataset

In [ ]:
df = pd.read_csv('Customer_segmentation_dataset.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nFirst 5 rows:')
df.head()

---
## 3. Data Exploration (EDA)

In [ ]:
print('Column data types:')
print(df.dtypes)
print()
print('Memory usage:')
df.info()

In [ ]:
print('Numeric summary:')
df.describe()

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}\n')
for col in cat_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Columns with missing values:')
display(missing_df)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(missing_df.index, missing_df['Missing %'], color='steelblue', edgecolor='white')
ax.set_title('Missing Data by Column (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Column')
ax.set_ylabel('Missing (%)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'ID']

fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    df[col].dropna().hist(ax=ax, bins=25, color='royalblue', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.suptitle('Distribution of Numeric Features', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plot_cats = [c for c in cat_cols if c != 'Var_1']
fig, axes = plt.subplots(1, len(plot_cats), figsize=(16, 4))
for ax, col in zip(axes, plot_cats):
    vc = df[col].value_counts(dropna=False)
    vc.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Feature Counts', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Data Preparation

Steps covered:
- Drop the `ID` column (non-predictive identifier)
- Remove duplicate rows
- Handle missing values

In [ ]:
df_clean = df.drop(columns=['ID'])
print(f'Shape after dropping ID: {df_clean.shape}')

In [ ]:
n_dup = df_clean.duplicated().sum()
print(f'Duplicate rows found: {n_dup}')
df_clean = df_clean.drop_duplicates()
print(f'Shape after removing duplicates: {df_clean.shape}')

In [ ]:
cat_fill_cols = ['Ever_Married', 'Graduated', 'Profession', 'Var_1']
num_fill_cols = ['Work_Experience', 'Family_Size']

for col in cat_fill_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f'  [{col}] filled NaN with mode → "{mode_val}"')

for col in num_fill_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f'  [{col}] filled NaN with median → {median_val}')

print(f'\nMissing values remaining: {df_clean.isnull().sum().sum()}')

---
## 5. Data Preprocessing — Encoding Categorical Variables



In [ ]:
df_encoded = df_clean.copy()

binary_map = {
    'Gender':       {'Male': 0, 'Female': 1},
    'Ever_Married': {'No': 0, 'Yes': 1},
    'Graduated':    {'No': 0, 'Yes': 1},
}
for col, mapping in binary_map.items():
    df_encoded[col] = df_encoded[col].map(mapping)
    print(f'  [{col}] encoded: {mapping}')

print()

In [ ]:
spending_map = {'Low': 0, 'Average': 1, 'High': 2}
df_encoded['Spending_Score'] = df_encoded['Spending_Score'].map(spending_map)
print(f'Spending_Score ordinal map: {spending_map}')
print(df_encoded['Spending_Score'].value_counts().sort_index())

In [ ]:
df_encoded = pd.get_dummies(df_encoded, columns=['Profession', 'Var_1'], drop_first=True)
print(f'Shape after one-hot encoding: {df_encoded.shape}')
print(f'\nColumns now: {list(df_encoded.columns)}')

In [ ]:
obj_remaining = df_encoded.select_dtypes(include='object').columns.tolist()
bool_cols     = df_encoded.select_dtypes(include='bool').columns.tolist()

df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f'Object columns remaining: {obj_remaining}')
print(f'Boolean columns converted to int: {bool_cols}')
df_encoded.head(3)

---
## 6. Outlier Detection & Removal

Using the **IQR (Interquartile Range)** method on continuous numeric features.  


In [ ]:
outlier_cols = ['Age', 'Work_Experience', 'Family_Size']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(12, 4))
for ax, col in zip(axes, outlier_cols):
    ax.boxplot(df_encoded[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col}\n(before removal)', fontweight='bold')
    ax.set_ylabel('Value')
plt.suptitle('Boxplots — Outlier Inspection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_no_outliers = df_encoded.copy()
rows_before = len(df_no_outliers)

for col in outlier_cols:
    Q1  = df_no_outliers[col].quantile(0.25)
    Q3  = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_no_outliers[col] < lower) | (df_no_outliers[col] > upper)).sum()
    df_no_outliers = df_no_outliers[
        (df_no_outliers[col] >= lower) & (df_no_outliers[col] <= upper)
    ]
    print(f'  [{col}] bounds: [{lower:.2f}, {upper:.2f}] | rows removed: {n_out}')

rows_after = len(df_no_outliers)
print(f'\nRows before: {rows_before:,} → after: {rows_after:,} (removed {rows_before - rows_after})')

In [ ]:
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(12, 4))
for ax, col in zip(axes, outlier_cols):
    ax.boxplot(df_no_outliers[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightgreen'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col}\n(after removal)', fontweight='bold')
    ax.set_ylabel('Value')
plt.suptitle('Boxplots — After Outlier Removal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Feature Engineering

Creating new informative features derived from existing columns to improve clustering quality:


In [ ]:
df_fe = df_no_outliers.copy()

bins   = [0, 25, 35, 50, 65, 100]
labels = [0, 1, 2, 3, 4]
df_fe['Age_Group'] = pd.cut(df_fe['Age'], bins=bins, labels=labels).astype(int)
print('Age_Group distribution:')
age_label_map = {0:'Young(≤25)', 1:'YoungAdult(26-35)', 2:'MiddleAged(36-50)',
                 3:'Senior(51-65)', 4:'Elderly(65+)'}
print(df_fe['Age_Group'].map(age_label_map).value_counts())
print()

In [ ]:
df_fe['Experience_per_Age'] = (df_fe['Work_Experience'] / df_fe['Age']).round(4)
print('Experience_per_Age — sample stats:')
print(df_fe['Experience_per_Age'].describe())
print()

In [ ]:
df_fe['Is_Senior'] = (df_fe['Age'] >= 60).astype(int)
print(f'Is_Senior counts:\n{df_fe["Is_Senior"].value_counts()}')
print()

In [ ]:
df_fe['Affluence_Score'] = df_fe['Spending_Score'] + df_fe['Graduated']
print('Affluence_Score distribution:')
print(df_fe['Affluence_Score'].value_counts().sort_index())
print()

In [ ]:
df_fe['Dependents_Flag'] = (df_fe['Family_Size'] > 3).astype(int)
print(f'Dependents_Flag counts:\n{df_fe["Dependents_Flag"].value_counts()}')
print()

print(f'Total features after engineering: {df_fe.shape[1]}')
print(f'Feature list: {list(df_fe.columns)}')

In [ ]:
num_feats = df_fe.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_fe[num_feats].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix (after Feature Engineering)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Data Transformation & Feature Scaling

Scaling is critical for distance-based clustering algorithms (K-Means, DBSCAN, etc.).  
We compare three scalers and apply **RobustScaler** (best for moderate outlier presence):


In [ ]:
scale_cols = ['Age', 'Work_Experience', 'Family_Size',
              'Spending_Score', 'Experience_per_Age', 'Affluence_Score']

no_scale_cols = [c for c in df_fe.columns if c not in scale_cols]
print(f'Columns to scale     : {scale_cols}')
print(f'Columns kept as-is   : {no_scale_cols}')

In [ ]:
demo_col = 'Age'

std_vals    = StandardScaler().fit_transform(df_fe[[demo_col]])
minmax_vals = MinMaxScaler().fit_transform(df_fe[[demo_col]])
robust_vals = RobustScaler().fit_transform(df_fe[[demo_col]])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
data_labels = [
    (df_fe[demo_col].values, 'Original (Age)', 'steelblue'),
    (std_vals.ravel(),       'StandardScaler', 'coral'),
    (minmax_vals.ravel(),    'MinMaxScaler',   'mediumseagreen'),
    (robust_vals.ravel(),    'RobustScaler',   'mediumpurple'),
]
for ax, (vals, title, color) in zip(axes, data_labels):
    ax.hist(vals, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.suptitle('Scaling Comparison — Age Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_scaled = df_fe.copy()
scaler    = RobustScaler()

df_scaled[scale_cols] = scaler.fit_transform(df_fe[scale_cols])

print('Scaled feature statistics (should have near-zero median):')
display(df_scaled[scale_cols].describe().T[
    ['mean', 'std', 'min', '50%', 'max']
].rename(columns={'50%': 'median'}).round(4))

---
## 9. Train / Test Split

Although Customer Segmentation is an **unsupervised** task (clustering):
- Evaluating cluster stability on unseen data
- Validating that segment assignments generalise

In [ ]:
train_df, test_df = train_test_split(df_scaled, test_size=0.2, random_state=42)

print(f'Training set : {train_df.shape[0]:,} rows  ({train_df.shape[0]/len(df_scaled)*100:.1f}%)')
print(f'Test set     : {test_df.shape[0]:,}  rows  ({test_df.shape[0]/len(df_scaled)*100:.1f}%)')
print(f'Features     : {df_scaled.shape[1]}')

---
## 10. Save Processed Datasets

In [ ]:
df_scaled.to_csv('customer_segmentation_processed.csv', index=False)
train_df.to_csv('customer_seg_train.csv', index=False)
test_df.to_csv('customer_seg_test.csv', index=False)

print('Files saved:')
print('  customer_segmentation_processed.csv  — full processed dataset')
print('  customer_seg_train.csv               — 80% training split')
print('  customer_seg_test.csv                — 20% test split')